### Column lineage builder (data dictionary → lineage graph)

This notebook builds **column-level semantic lineage** from an Excel data dictionary using:
- Candidate generation via text similarity (TF‑IDF)
- Optional **LLM confirmation** to label/directionally link columns

**Required columns in the XLSX** (or provide a mapping):
- `table_name`
- `column_name`
- `business attribute name`
- `description`

Outputs are written to `out_lineage/`:
- `lineage_edges.csv`
- `lineage_edges.json`
- `debug_per_target.json` (review candidates + raw model JSON)
- `llm_cache.jsonl` (caches model responses)


In [ ]:
# Install dependencies (run once per environment)
import sys

!{sys.executable} -m pip install -r "requirements-lineage.txt"


In [ ]:
from pathlib import Path

# --- XLSX location ---
# You said `ikg data dictionary.xlsx` is uploaded in the SAME folder as this notebook.
# So the default is a relative path that should "just work".

xlsx_path = Path("ikg data dictionary.xlsx")

# If you want to point to a different path, uncomment one of these:
# Windows example:
# xlsx_path = Path(r"c:\dev\ikg data dictionary.xlsx")
# Linux example:
# xlsx_path = Path("/workspace/ikg data dictionary.xlsx")

if not xlsx_path.exists():
    raise FileNotFoundError(
        f"Could not find '{xlsx_path}'. Put the file next to this notebook or set xlsx_path explicitly."
    )

# Optional: if your sheet isn't the first one
# Use 0 for the first sheet, or a string sheet name like "Sheet1".
sheet_name = 0

# Optional: exact Excel header mapping (only needed if auto-detect fails)
column_mapping = None
# column_mapping = {
#     "table_name": "table_name",
#     "column_name": "column_name",
#     "business_attribute_name": "business attribute name",
#     "description": "description",
# }

# Where to write outputs
out_dir = Path("out_lineage")
out_dir.mkdir(parents=True, exist_ok=True)

xlsx_path.resolve()


In [ ]:
import os

# --- LLM config ---
# Option A: OpenAI (recommended)
llm_provider = "openai"  # "openai" | "ollama" | "none"
llm_model = "gpt-4.1-mini"

# Make sure your key is set in the environment for OpenAI:
# os.environ["OPENAI_API_KEY"] = "..."

# Option B: Ollama (local)
ollama_base_url = "http://localhost:11434"

# Candidate generation controls
candidates_cfg = {
    "top_k": 20,
    "min_score": 0.15,
    "include_same_table": False,
    "max_edges_per_target": 2,
}


In [ ]:
import yaml

# Build an in-memory config dict (same structure as lineage_config.example.yaml)
config = {
    "input": {
        "path": str(xlsx_path),
        "sheet_name": sheet_name,
        "column_mapping": column_mapping,
    },
    "output": {"dir": str(out_dir)},
    "candidates": candidates_cfg,
    "llm": {
        "provider": llm_provider,
        "model": llm_model,
        "base_url": ollama_base_url,
    },
}

# Save config alongside outputs for reproducibility
config_path = out_dir / "lineage_config.used.yaml"
config_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")

config_path


In [ ]:
# Run the pipeline
from build_column_lineage import run

run(config)


In [ ]:
import pandas as pd

edges_csv = out_dir / "lineage_edges.csv"
edges_json = out_dir / "lineage_edges.json"

df_edges = pd.read_csv(edges_csv)
print("Edges:", len(df_edges))
df_edges.head(20)


### Notes / interpretation

- This builds **semantic** lineage (meaning-based) from metadata.
- If you set `llm_provider = "none"`, you’ll get similarity candidates only (good for initial triage).
- For higher precision, keep `top_k` modest (10–30) and review `debug_per_target.json`.
